## Example 4 — Improving RAG with Hybrid Search and Empty Retrievals

**Goal:** Build a small digital-library knowledge base and compare semantic retrieval, BM25 keyword retrieval, and hybrid search using Reciprocal Rank Fusion (RRF). The example also demonstrates a fallback when the knowledge base does not contain an answer.

In [1]:
!pip -q install langchain-text-splitters chromadb sentence-transformers rank_bm25 pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions
from rank_bm25 import BM25Okapi
import pandas as pd

documents = {
    "digital_library.txt": """Digital Library Guide

A digital library organizes electronic books, articles, and learning resources so students can search them from a computer.
Good search tools help readers find resources by title, subject, author, or the meaning of a question.
Study planning works best when students break large assignments into smaller tasks and schedule focused study sessions.
Short breaks between study sessions can help maintain attention and reduce fatigue.""",

    "learning_resources.txt": """Learning Resources and Study Skills

Programming tutorials teach coding concepts through explanations, examples, and exercises.
Reference books are useful when a learner needs detailed explanations or wants to revisit a topic.
A weekly study plan can balance lectures, assignments, revision, and practice.
Students can improve understanding by explaining a concept in their own words and solving practice questions.""",

    "library_catalog.txt": """Library Catalog Notes

The book Python Basics for Beginners is listed under library item code LIB-7304 and covers variables, conditions, loops, and functions.
The book Data Structures Made Simple is listed under library item code LIB-8852 and introduces arrays, stacks, queues, and trees.
Library requests should include the exact item code so staff can locate the correct book quickly."""
}

splitter = RecursiveCharacterTextSplitter(
    chunk_size=220,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""]
)

client_rag = chromadb.Client()
embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
collection_rag = client_rag.create_collection(
    name="assignment_library_hybrid",
    embedding_function=embedder
)

all_chunks, all_chunk_ids, all_chunk_meta = [], [], []
for source, text in documents.items():
    chunks = splitter.split_text(text)
    ids = [f"{source}_chunk_{i}" for i in range(len(chunks))]
    metadata = [{"source": source, "chunk_index": i} for i in range(len(chunks))]
    collection_rag.add(documents=chunks, metadatas=metadata, ids=ids)
    all_chunks.extend(chunks)
    all_chunk_ids.extend(ids)
    all_chunk_meta.extend(metadata)

print("Total chunks stored:", collection_rag.count())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total chunks stored: 11


In [3]:
def semantic_search(query, n_results=3):
    results = collection_rag.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )
    output = []
    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        output.append({
            "text": doc,
            "source": meta["source"],
            "similarity": round(1 - distance, 4)
        })
    return output

for result in semantic_search("LIB-7304", n_results=3):
    print(f"[{result['similarity']}] ({result['source']}) {result['text']}")

[0.2711] (library_catalog.txt) The book Python Basics for Beginners is listed under library item code LIB-7304 and covers variables, conditions, loops, and functions.
[0.2243] (library_catalog.txt) The book Data Structures Made Simple is listed under library item code LIB-8852 and introduces arrays, stacks, queues, and trees.
[0.2008] (library_catalog.txt) Library Catalog Notes


In [4]:
def tokenize(text):
    return text.lower().replace("-", " ").split()

bm25 = BM25Okapi([tokenize(chunk) for chunk in all_chunks])

def bm25_search(query, n_results=3):
    scores = bm25.get_scores(tokenize(query))
    ranked_indices = sorted(
        range(len(scores)), key=lambda i: scores[i], reverse=True
    )[:n_results]
    output = []
    for idx in ranked_indices:
        if scores[idx] > 0:
            output.append({
                "text": all_chunks[idx],
                "source": all_chunk_meta[idx]["source"],
                "bm25_score": round(float(scores[idx]), 4)
            })
    return output

for result in bm25_search("LIB-7304", n_results=3):
    print(f"[{result['bm25_score']}] ({result['source']}) {result['text']}")

[2.9588] (library_catalog.txt) The book Python Basics for Beginners is listed under library item code LIB-7304 and covers variables, conditions, loops, and functions.
[1.2039] (library_catalog.txt) The book Data Structures Made Simple is listed under library item code LIB-8852 and introduces arrays, stacks, queues, and trees.


In [5]:
def hybrid_search(query, n_results=3, k=60):
    semantic_results = semantic_search(query, n_results=10)
    keyword_results = bm25_search(query, n_results=10)
    rrf_scores, chunk_lookup = {}, {}

    for rank, result in enumerate(semantic_results):
        key = result["text"]
        rrf_scores[key] = rrf_scores.get(key, 0) + 1 / (k + rank + 1)
        chunk_lookup[key] = result

    for rank, result in enumerate(keyword_results):
        key = result["text"]
        rrf_scores[key] = rrf_scores.get(key, 0) + 1 / (k + rank + 1)
        chunk_lookup.setdefault(key, result)

    ranked = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)[:n_results]
    return [
        {
            "text": text,
            "source": chunk_lookup[text]["source"],
            "rrf_score": round(score, 5)
        }
        for text, score in ranked
    ]

for result in hybrid_search("LIB-7304 book details", n_results=3):
    print(f"[{result['rrf_score']}] ({result['source']}) {result['text']}")

[0.03227] (library_catalog.txt) Library requests should include the exact item code so staff can locate the correct book quickly.
[0.03227] (library_catalog.txt) The book Python Basics for Beginners is listed under library item code LIB-7304 and covers variables, conditions, loops, and functions.
[0.03175] (library_catalog.txt) The book Data Structures Made Simple is listed under library item code LIB-8852 and introduces arrays, stacks, queues, and trees.


In [6]:
query = "What is LIB-7304?"
semantic_top = semantic_search(query, n_results=1)
keyword_top = bm25_search(query, n_results=1)
hybrid_top = hybrid_search(query, n_results=1)

comparison = pd.DataFrame({
    "Method": ["Semantic only", "Keyword (BM25) only", "Hybrid (RRF)"],
    "Top result": [
        semantic_top[0]["text"][:90] + "..." if semantic_top else "(no result)",
        keyword_top[0]["text"][:90] + "..." if keyword_top else "(no result)",
        hybrid_top[0]["text"][:90] + "..." if hybrid_top else "(no result)"
    ]
})
display(comparison)

,Method,Top result
0,Semantic only,The book Python Basics for Beginners is listed...
1,Keyword (BM25) only,The book Python Basics for Beginners is listed...
2,Hybrid (RRF),The book Python Basics for Beginners is listed...


In [7]:
def retrieve_with_threshold(query, n_results=5, similarity_threshold=0.3):
    results = collection_rag.query(
        query_texts=[query],
        n_results=min(n_results, collection_rag.count()),
        include=["documents", "metadatas", "distances"]
    )
    filtered = []
    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        similarity = 1 - distance
        if similarity >= similarity_threshold:
            filtered.append({
                "text": doc,
                "source": meta["source"],
                "similarity": round(similarity, 4)
            })
    return filtered

KNOWLEDGE_BASE_TOPICS = sorted({meta["source"] for meta in all_chunk_meta})

def smart_retrieve(query, n_results=4, similarity_threshold=0.3):
    primary = retrieve_with_threshold(
        query, n_results=n_results, similarity_threshold=similarity_threshold
    )
    if primary:
        return {"status": "ok", "method": "semantic", "chunks": primary}

    fallback = hybrid_search(query, n_results=n_results)
    if fallback:
        return {"status": "ok", "method": "hybrid_fallback", "chunks": fallback}

    return {
        "status": "empty",
        "method": None,
        "chunks": [],
        "message": (
            "I don't know based on the provided documents. "
            f"This knowledge base covers: {', '.join(KNOWLEDGE_BASE_TOPICS)}."
        )
    }

for label, question in [
    ("Test 1", "How can students organize study time?"),
    ("Test 2", "Tell me about LIB-7304"),
    ("Test 3", "How do I repair a flat tire?")
]:
    result = smart_retrieve(question)
    print(label, "->", result["status"], "|", result.get("method") or result.get("message"))

Test 1 -> ok | semantic
Test 2 -> ok | semantic
Test 3 -> ok | hybrid_fallback


In [8]:
def format_context_for_llm(query, chunks):
    context = "\n\n".join(
        f"[Source: {chunk['source']}]\n{chunk['text']}" for chunk in chunks
    )
    return f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"

def safe_rag_prompt(query):
    retrieval = smart_retrieve(query)
    if retrieval["status"] == "empty":
        return retrieval["message"]
    prompt = format_context_for_llm(query, retrieval["chunks"])
    return f"[Prompt prepared using {retrieval['method']} search]\n\n{prompt}"

print(safe_rag_prompt("What is LIB-7304?"))
print("\n" + "=" * 65 + "\n")
print(safe_rag_prompt("How do I repair a flat tire?"))

[Prompt prepared using hybrid_fallback search]

Context:
[Source: library_catalog.txt]
The book Python Basics for Beginners is listed under library item code LIB-7304 and covers variables, conditions, loops, and functions.

[Source: library_catalog.txt]
The book Data Structures Made Simple is listed under library item code LIB-8852 and introduces arrays, stacks, queues, and trees.

[Source: library_catalog.txt]
Library requests should include the exact item code so staff can locate the correct book quickly.

[Source: library_catalog.txt]
Library Catalog Notes

Question: What is LIB-7304?

Answer:


[Prompt prepared using hybrid_fallback search]

Context:
[Source: learning_resources.txt]
A weekly study plan can balance lectures, assignments, revision, and practice.
Students can improve understanding by explaining a concept in their own words and solving practice questions.

[Source: digital_library.txt]
A digital library organizes electronic books, articles, and learning resources so st